# 6b. Doladenie (fine-tuning) siete R(2+1)D

V notebooku 6 je sieť R(2+1)D zmrazená a trénuje sa iba SVM nad jej príznakmi.
Tu sa trénuje aj časť siete:
- poslednú vrstvu nahradíme novou s jedným výstupom pre každú triedu,
- trénujeme novú vrstvu a posledný blok siete (`layer4`); ostatné vrstvy ostávajú
  s váhami z Kinetics-400.

Nastavenia trénovania:
- malý krok učenia (`LR`) a dropout pred výstupnou vrstvou,
- náhodné zrkadlové otočenie klipov (augmentácia),
- early stopping podľa validačnej straty (ako pri LSTM v notebooku 5),
- vrstvy BatchNorm ostávajú v režime inferencie (dávka 4 klipov je na odhad ich štatistík malá).

Hodnotenie je rovnaké ako v notebooku 4. Jedno delenie trvá na GPU niekoľko minút;
pre rýchly test možno znížiť `N_SPLITS`.

In [ ]:
import copy

import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

from har.config import CLASSES, INDEX_FILE
from har.data import read_frames, to_video_tensor
from har.evaluation import (make_splits, plot_confusion, plot_history, predictions_table,
                            save_predictions, summary, train_val_split)
from har.models import class_weights

N_SPLITS = 10
EPOCHS = 8
BATCH = 4
LR = 1e-5
TRAIN_LAYERS = ("layer4", "fc")      # trénované časti siete

df = pd.read_csv(INDEX_FILE)
y = df.label.values
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"           # výpočty vo float16 šetria pamäť GPU

## Príprava klipov

Predspracovanie je rovnaké ako v notebooku 3. Klipy pripravíme raz a držíme v pamäti,
aby sa snímky nenačítavali v každej epoche.

In [ ]:
weights = R2Plus1D_18_Weights.KINETICS400_V1
video_transform = weights.transforms()
pretrained = r2plus1d_18(weights=weights).state_dict()

clips = torch.stack([to_video_tensor(read_frames(p), video_transform)[0].half()
                     for p in df.path])                    # (klipy, 3, 16, 112, 112)
print(clips.shape)

## Model a trénovanie

In [ ]:
def build_model():
    model = r2plus1d_18()
    model.load_state_dict(pretrained)
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(512, len(CLASSES)))
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith(TRAIN_LAYERS)
    return model.to(device)


def freeze_batchnorm(model):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm3d):
            m.eval()


def predict_logits(model, idx):
    model.eval()
    out = []
    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        for b in np.array_split(idx, max(1, len(idx) // 8)):
            out.append(model(clips[b].to(device).float()).float().cpu())
    return torch.cat(out)

In [ ]:
def train_finetune(i_tr, i_va, seed=0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = build_model()
    criterion = nn.CrossEntropyLoss(weight=class_weights(y[i_tr]).to(device))
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                                 lr=LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(enabled=use_amp)
    y_va = torch.tensor(y[i_va], device=device)

    history, best_loss, best_epoch, best_state = [], np.inf, 0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        freeze_batchnorm(model)
        train_loss = 0
        for b in np.array_split(rng.permutation(i_tr), max(1, len(i_tr) // BATCH)):
            x = clips[b].to(device).float()
            flip = torch.rand(len(b)) < 0.5
            x[flip] = x[flip].flip(-1)                      # zrkadlové otočenie
            yb = torch.tensor(y[b], device=device)

            optimizer.zero_grad()
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(x), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * len(b)

        val_loss = criterion(predict_logits(model, i_va).to(device), y_va).item()
        history.append((train_loss / len(i_tr), val_loss))
        if val_loss < best_loss:
            best_loss, best_epoch, best_state = val_loss, epoch, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, history, best_epoch

## Hodnotenie

In [ ]:
tables, first_history = [], None
for split, tr, te in make_splits(df, n_splits=N_SPLITS):
    i_tr, i_va = train_val_split(df, tr, split)
    model, history, best_epoch = train_finetune(i_tr, i_va, seed=split)
    y_pred = predict_logits(model, te).argmax(1).numpy()

    tables.append(predictions_table(split, te, y[te], y_pred))
    if first_history is None:
        first_history = (history, best_epoch)
    print(f"delenie {split}: presnosť {(y_pred == y[te]).mean():.0%}, vybraná epocha {best_epoch}")

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

pred = pd.concat(tables, ignore_index=True)
history, best_epoch = first_history
save_predictions(pred, "video_finetune")
summary(pred, "R(2+1)D doladená")

## Priebeh trénovania

Strata na trénovacích a validačných dátach v prvom delení.
Zvislá čiara označuje epochu, ktorej váhy model používa.

In [ ]:
plot_history(history, best_epoch)

## Chyby modelu

Matica zámen (*confusion matrix*) zo všetkých 10 delení. Riadok = skutočná trieda,
stĺpec = predpoveď modelu.
- vľavo: počet testovacích klipov (súčet cez 10 delení, každý klip je v teste viackrát),
- vpravo: % klipov danej skutočnej triedy (súčet v riadku je 100 %).

In [ ]:
plot_confusion(pred, "R(2+1)D doladená")